# Checking the device

In [1]:
import torch

print("Is a ROCm-GPU detected? ", torch.cuda.is_available())
print("How many ROCm-GPUs are detected? ", torch.cuda.device_count())

Is a ROCm-GPU detected?  True
How many ROCm-GPUs are detected?  1


/opt/conda/envs/py_3.12/lib/python3.12/site-packages/torch/cuda/__init__.py:736: UserWarning: Can't initialize amdsmi - Error code: 34
  warnings.warn(f"Can't initialize amdsmi - Error code: {e.err_code}")


In [2]:
import gc
gc.collect()
torch.cuda.empty_cache()

# Library setup

In [3]:
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model
import os
import datetime

datetime_now = datetime.datetime.now()

/opt/conda/envs/py_3.12/lib/python3.12/site-packages/redis/connection.py:77: UserWarning: redis-py works best with hiredis. Please consider installing
  warnings.warn(msg)


# Dataset preparation

In [4]:
os.getcwd()

'/workspace/opendrive_generation'

In [5]:
# Load dataset from JSONL
dataset = load_dataset(
    "json",
    data_files="/workspace/opendrive_generation/xodr_generated_scenarios_20250624_161625/scenario_metadata.jsonl",
    split="train",
)

# Shuffle for training variety
dataset = dataset.shuffle(seed=42)


# Combine prompt and response into a training text field
def format_example(example):
    prompt = example.get("prompt", "").strip()

    if "script_path" not in example:
        raise RuntimeError("No script_path in the json entry.")

    script_path = example.get("script_path", "")
    code = ""

    with open(f"{os.getcwd()}/{script_path}", "r") as code_file:
        code = code_file.read().strip()

    response = example.get("response", "").strip()
    return {
        "text": f"### Prompt:\n{prompt}\n\n### Response:\n{code}",
        "prompt": prompt,
        "response": code,
    }


# Apply mapping
dataset = dataset.map(format_example)

# Example check
print(dataset[0]["text"])

### Prompt:
Start with a straight road and gently curve it away using a spiral shape.

### Response:
from scenariogeneration import xodr, prettyprint, ScenarioGenerator

class Scenario(ScenarioGenerator):
    def __init__(self):
        super().__init__()

    def road(self):
        road = xodr.create_road(
            [xodr.Line(30), xodr.Spiral(-0.00001, -0.035, 200)],
            id=2,
            left_lanes=2,
            right_lanes=2
        )
        odr = xodr.OpenDrive("line_spiral_combo")
        odr.add_road(road)
        odr.adjust_roads_and_lanes()
        return odr

if __name__ == "__main__":
    sce = Scenario()
    prettyprint(sce.road().get_element())
    sce.generate(".")


# Model and tokenizer

In [ ]:
# Load base model to GPU memory.
device = "cuda:0"
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
import torch


def load_model_and_tokenizer(
    base_model_name: str,
    adapter_path: str = None,
    special_tokens: dict = {"additional_special_tokens": ["<|endofcode|>"]},
    use_4bit: bool = True,
    compute_dtype: torch.dtype = torch.float16,
):
    # BitsAndBytes config for 4-bit loading
    quant_config = BitsAndBytesConfig(
        load_in_4bit=use_4bit,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=compute_dtype,
        bnb_4bit_quant_type="nf4",
    )

    print("🔄 Loading base model...")
    model = AutoModelForCausalLM.from_pretrained(
        base_model_name,
        device_map="auto",
        quantization_config=quant_config,
        trust_remote_code=True,
    )

    print("🔠 Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True)

    # Add special tokens
    if special_tokens:
        tokenizer.add_special_tokens(special_tokens)
        model.resize_token_embeddings(len(tokenizer))

    # Fix missing pad token by reusing eos
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Load LoRA adapter if provided
    if adapter_path:
        print("🔁 Loading PEFT adapter...")
        model = PeftModel.from_pretrained(model, adapter_path)
        model.eval()

    return model, tokenizer


model_name = "mistralai/Mistral-7B-Instruct-v0.3"
adapter_path = f"llm_xodr_finetuned_model_{datetime_now.strftime("%Y_%m_%d_%H_%M_%S")}_{model_name}"

model, tokenizer = load_model_and_tokenizer(
    base_model_name=model_name, adapter_path=None
)

🔄 Loading base model...


/opt/conda/envs/py_3.12/lib/python3.12/site-packages/torch/cuda/__init__.py:736: UserWarning: Can't initialize amdsmi - Error code: 34
  warnings.warn(f"Can't initialize amdsmi - Error code: {e.err_code}")


g++ (Ubuntu 13.3.0-6ubuntu2~24.04) 13.3.0
Copyright (C) 2023 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.



Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

/opt/conda/envs/py_3.12/lib/python3.12/site-packages/redis/connection.py:77: UserWarning: redis-py works best with hiredis. Please consider installing
  warnings.warn(msg)
/opt/conda/envs/py_3.12/lib/python3.12/site-packages/torch/cuda/__init__.py:736: UserWarning: Can't initialize amdsmi - Error code: 34
  warnings.warn(f"Can't initialize amdsmi - Error code: {e.err_code}")


In [ ]:
lengths = []

for example in dataset:
    text = f"### Prompt:\n{example['prompt']}\n\n### Response:\n{example['response']}"
    tokens = tokenizer(text)["input_ids"]
    lengths.append(len(tokens))

import numpy as np

print(
    f"Mean: {np.mean(lengths):.1f}, Median: {np.median(lengths)}, 95th percentile: {np.percentile(lengths, 95)}"
)

Mean: 381.9, Median: 312.0, 95th percentile: 694.0


In [ ]:
config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, config)
model.print_trainable_parameters()

trainable params: 13,631,488 || all params: 7,261,663,232 || trainable%: 0.1877


In [ ]:
# def tokenize(example):
#     prompt = f"### Prompt:\n{example['prompt']}\n\n### Response:\n"
#     response = example["response"].strip() + "\n<|endofcode|>"

#     full_text = prompt + response
#     tokenized = tokenizer(full_text, padding="max_length", truncation=True, max_length=512)

#     # Mask out the prompt part so loss is computed only over code
#     prompt_len = len(tokenizer(prompt)["input_ids"])
#     labels = tokenized["input_ids"].copy()
#     labels[:prompt_len] = [-100] * prompt_len

#     tokenized["labels"] = labels
#     return tokenized

# def tokenize(batch):
#     prompts = [f"### Prompt:\n{p}\n\n### Response:\n" for p in batch["prompt"]]
#     responses = batch["response"]
#     full_texts = [prompt + response for prompt, response in zip(prompts, responses)]

#     encodings = tokenizer(
#         full_texts, padding="max_length", truncation=True, max_length=512
#     )

#     # Mask the prompt section in the labels
#     labels = []
#     for i in range(len(prompts)):
#         label = encodings["input_ids"][i].copy()
#         prompt_len = len(tokenizer(prompts[i])["input_ids"])
#         label[:prompt_len] = [-100] * prompt_len
#         labels.append(label)

#     encodings["labels"] = labels
#     return encodings


def tokenize(batch):
    prompts = [f"### Prompt:\n{p}\n\n### Response:\n" for p in batch["prompt"]]
    responses = [r.strip() + "\n<|endofcode|>" for r in batch["response"]]

    full_texts = [p + r for p, r in zip(prompts, responses)]

    encodings = tokenizer(
        full_texts,
        padding="max_length",
        truncation=True,
        max_length=512,
        return_tensors="pt",
    )

    input_ids = encodings["input_ids"]
    labels = input_ids.clone()

    for i, prompt_text in enumerate(prompts):
        prompt_ids = tokenizer(prompt_text, truncation=True, max_length=512)[
            "input_ids"
        ]
        prompt_len = len(prompt_ids)
        labels[i][:prompt_len] = -100  # mask out prompt tokens

    encodings["labels"] = labels
    return {
        k: v.tolist() for k, v in encodings.items()
    }  # convert tensors to lists for HuggingFace compatibility


tokenizer.pad_token = tokenizer.eos_token
tokenized = dataset.map(tokenize, batched=True)

# Fine Tuning loop

In [ ]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer, mlm=False  # we're doing causal LM, not masked LM
)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir llm_xodr_finetuned/runs

ERROR: Failed to launch TensorBoard (exited with 1).
Contents of stderr:
TensorFlow installation not found - running with reduced feature set.

NOTE: Using experimental fast data loading logic. To disable, pass
    "--load_fast=false" and report issues on GitHub. More details:
    https://github.com/tensorflow/tensorboard/issues/4784

Traceback (most recent call last):
  File "/opt/conda/envs/py_3.12/bin/tensorboard", line 8, in <module>
    sys.exit(run_main())
             ^^^^^^^^^^
  File "/opt/conda/envs/py_3.12/lib/python3.12/site-packages/tensorboard/main.py", line 46, in run_main
    app.run(tensorboard.main, flags_parser=tensorboard.configure)
  File "/opt/conda/envs/py_3.12/lib/python3.12/site-packages/absl/app.py", line 316, in run
    _run_main(main, args)
  File "/opt/conda/envs/py_3.12/lib/python3.12/site-packages/absl/app.py", line 261, in _run_main
    sys.exit(main(argv))
             ^^^^^^^^^^
  File "/opt/conda/envs/py_3.12/lib/python3.12/site-packages/tensorboard/p

In [ ]:
from transformers import TrainerCallback
import torch


class PromptLoggingCallback(TrainerCallback):
    def __init__(self, tokenizer, dataset, interval=50):
        self.tokenizer = tokenizer
        self.dataset = dataset
        self.interval = interval

    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step % self.interval == 0:
            model = kwargs["model"]
            prompt = self.dataset[state.global_step % len(self.dataset)]["prompt"]
            formatted = f"### Prompt:\n{prompt}\n\n### Response:\n"
            input_ids = self.tokenizer(formatted, return_tensors="pt").to(model.device)

            model.eval()
            with torch.no_grad():
                outputs = model.generate(
                    **input_ids,
                    max_new_tokens=1000,
                    do_sample=False,
                    temperature=0.0,
                    pad_token_id=self.tokenizer.eos_token_id,
                )
            decoded = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
            response_only = decoded.replace(formatted, "").strip()

            print("\n" + "=" * 80)
            print(f"Step {state.global_step}")
            print("📥 Prompt:\n", prompt)
            print("📤 Model Output:\n", response_only)
            print("=" * 80 + "\n")

In [ ]:
def tune_batch_and_train(
    model,
    tokenizer,
    tokenized_dataset,
    data_collator,
    target_effective_batch_size=8,
    max_per_device_batch_size=8,
    prompt_logger=None,
    base_args=None
):
    if base_args is None:
        base_args = {
            "output_dir": "./llm_xodr_finetuned",
            "fp16": True,
            "label_names": ["labels"],
            "num_train_epochs": 1,
        }

    print(f"🔍 Auto-tuning batch size for target effective batch size ≈ {target_effective_batch_size}...\n")

    max_valid = 1
    for bs in range(1, max_per_device_batch_size + 1):
        try:
            gc.collect()
            torch.cuda.empty_cache()

            print(f"🧪 Trying per_device_train_batch_size = {bs}")

            trial_args = TrainingArguments(
                **base_args,
                per_device_train_batch_size=bs,
                gradient_accumulation_steps=1,
                max_steps=1,
                save_strategy="no",
                logging_steps=1,
                report_to="none"
            )

            trial_trainer = Trainer(
                model=model,
                args=trial_args,
                train_dataset=tokenized_dataset.select(range(4)),
                tokenizer=tokenizer,
                data_collator=data_collator
            )
            trial_trainer.train()
            max_valid = bs

        except RuntimeError as e:
            if "out of memory" in str(e).lower():
                print(f"❌ OOM at batch size {bs}")
                break
            else:
                raise e

    # Final tuning values
    per_device_train_batch_size = max_valid
    gradient_accumulation_steps = max(1, target_effective_batch_size // per_device_train_batch_size)

    print(f"\n🎯 Final Config:")
    print(f"    - per_device_train_batch_size = {per_device_train_batch_size}")
    print(f"    - gradient_accumulation_steps = {gradient_accumulation_steps}")
    print(f"    - effective batch size ≈ {per_device_train_batch_size * gradient_accumulation_steps}\n")

    # Final TrainingArguments
    final_args = TrainingArguments(
        **base_args,
        per_device_train_batch_size=per_device_train_batch_size,
        gradient_accumulation_steps=gradient_accumulation_steps,
        logging_steps=50,
        save_strategy="epoch",
        report_to="tensorboard"
    )

    # Final trainer
    trainer = Trainer(
        model=model,
        args=final_args,
        train_dataset=tokenized_dataset,
        tokenizer=tokenizer,
        data_collator=data_collator,
        callbacks=[prompt_logger] if prompt_logger else None
    )

    return trainer

In [ ]:
from transformers import TrainingArguments, Trainer

prompt_logger = PromptLoggingCallback(tokenizer=tokenizer, dataset=dataset, interval=50)

args = TrainingArguments(
    output_dir=adapter_path,
    logging_steps=50,
    num_train_epochs=1,
    save_strategy="epoch",
    fp16=True,
    report_to="tensorboard",
    label_names=["labels"],  # <-- this line solves the warning
)

trainer = tune_batch_and_train(
    model=model,
    tokenizer=tokenizer,
    tokenized_dataset=tokenized,
    data_collator=data_collator,
    target_effective_batch_size=8,
    max_per_device_batch_size=6,  # be conservative on a 24GB GPU
    prompt_logger=prompt_logger,  # optional, logs live examples
    base_args=args
)

# trainer = Trainer(
#     model=model,
#     args=args,
#     train_dataset=tokenized,
#     tokenizer=tokenizer,
#     data_collator=data_collator,
#     callbacks=[prompt_logger],  # 👈 Add it here
# )

# trainer = Trainer(model=model, args=args, train_dataset=tokenized, tokenizer=tokenizer)
trainer.train()

/tmp/ipykernel_5308/2987913484.py:16: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


# Model storage

In [ ]:
import datetime

datetime_now = datetime.datetime.now()
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)

('llm_xodr_finetuned_model/tokenizer_config.json',
 'llm_xodr_finetuned_model/special_tokens_map.json',
 'llm_xodr_finetuned_model/chat_template.jinja',
 'llm_xodr_finetuned_model/tokenizer.json')

# Inference testing

In [ ]:
model.eval()
prompt = "Make a road that curves gently to the right."
inputs = tokenizer(f"### Prompt:\n{prompt}\n\n### Response:\n", return_tensors="pt").to(
    "cuda"
)
eos_token_id = tokenizer.convert_tokens_to_ids("<|endofcode|>")

outputs = model.generate(
    **inputs,
    max_new_tokens=512,
    eos_token_id=eos_token_id,
    pad_token_id=tokenizer.eos_token_id,
    do_sample=False,
    temperature=0.0,
)

output_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
output_code = output_text.replace(input_text, "").split("<|endofcode|>")[0].strip()

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

### Prompt:
Make a road that curves gently to the right.

### Response:
from scenariogeneration import xodr, prettyprint, ScenarioGenerator

class Scenario(ScenarioGenerator):
    def __init__(self):
        super().__init__()

    def road(self):
        road = xodr.create_road(
            [xodr.Line(30), xodr.Spiral(-0.00001, -0.035, 200)],
            id=2,
            left_lanes=2,
            right_lanes=2
        )
        odr = xodr.OpenDrive("line_spiral_combo")
        odr.add_road(road)
        odr.adjust_roads_and_lanes()
        return odr

if __name__ == "__main__":
    sce = Scenario()
    prettyprint(sce.road().get_element())
    sce.generate(".")
```

Now you can run this script to generate the road layout using OpenDrive and some predefined geometry. Let me know if you have any questions or concerns!
